## ORN overlap

In [2]:
import pandas as pd
import plotly.graph_objects as go

yeast_orns = pd.read_csv('data/yeast_orns.csv')['root_id']
given_neurons = pd.read_csv('data/given_neurons.csv')
ovidns = given_neurons[given_neurons['group'] == "OviDN"]['root_id']
ca = given_neurons[given_neurons['group'] == "CA"]['root_id']
edge_list = pd.read_csv('out/master_edge_list_with_direction.csv')

orn_ovidn_edges = edge_list[edge_list['direction'] == 'ORN_to_oviDN']
orn_ca_edges = edge_list[edge_list['direction'] == 'ORN_to_CA']

orns_ovidns_only = set(orn_ovidn_edges[orn_ovidn_edges['pre_root_id'].isin(yeast_orns)]['pre_root_id'].unique())
orns_ca_only = set(orn_ca_edges[orn_ca_edges['pre_root_id'].isin(yeast_orns)]['pre_root_id'].unique())
orns_both = orns_ovidns_only & orns_ca_only
orns_neither = set(yeast_orns.unique()) - orns_ovidns_only - orns_ca_only - orns_both
orns_ovidns_only = orns_ovidns_only - orns_both
orns_ca_only = orns_ca_only - orns_both

# Prepare data for the pie chart
labels = ['ORN to OviDN only', 'ORN to CA only', 'Both', 'Neither']
values = [len(orns_ovidns_only), len(orns_ca_only), len(orns_both), len(orns_neither)]

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=0)])

fig.update_layout(
    title="Overlap of Yeast-Responding ORN Circuits",
    annotations=[dict(text="", x=0.5, y=0.5, font_size=20, showarrow=False)]
)

fig.show()

In [3]:
attrs = pd.read_csv('data/neurons.csv')
attrs = attrs[attrs['Root ID'].isin(yeast_orns)].fillna("None")
print(attrs.groupby('Sub Class')['Root ID'].count())
labels = attrs['Sub Class'].fillna('None').unique().tolist()
print(labels)
values = [len(attrs[attrs['Sub Class'] == l]) for l in labels]
fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=0)])

fig.update_layout(
    title="Sub classes of yeasty ORNs",
    annotations=[dict(text="", x=0.5, y=0.5, font_size=20, showarrow=False)]
)

fig.show()

Sub Class
antenna_olfactory_receptor_neuron           486
maxillary_palp_olfactory_receptor_neuron     76
Name: Root ID, dtype: int64
['antenna_olfactory_receptor_neuron', 'maxillary_palp_olfactory_receptor_neuron']


## Enrichment analysis as a bar chart

In [67]:
neurons = pd.read_csv('data/neurons.csv')
neurons['Predicted NT type'] = neurons['Predicted NT type'].fillna("Unknown")
master_edge_list = pd.read_csv('out/master_edge_list_with_direction.csv')

# 1. Pre-calculate global variables and force them to standard Python ints
total_neurons = int(len(neurons))

directions = {
    "ORN_to_CA": "Yeast ORNs to CA",
    "ORN_to_oviDN": "Yeast ORNs to OviDNs",
    "oviDN_to_CA": "OviDNs to CA",
    "CA_to_oviDN": "CA to OviDNs"
}

nt_info = {
    "DA": {"name": "Dopamine", "color": "#B87969"},
    "SER": {"name": "Serotonin", "color": "#8C6295"},
    "ACH": {"name": "Acetylcholine", "color": "#95A3CE"},
    "GABA": {"name": "GABA", "color": "#D5A848"},
    "GLUT": {"name": "Glutamate", "color": "#86A859"},
    "HIST": {"name": "Histamine", "color": "#C77C8A"},
    "TYR": {"name": "Tyrosine", "color": "#6FA7A3"},
    "OCT": {"name": "Octopamine", "color": "#725C98"},
    "Unknown": {"name": "Unknown", "color": "#CACACA"}
}

def enrichment_analysis():
    # Get unique NT types from the data
    nt_types = neurons['Predicted NT type'].fillna("Unknown").unique().tolist()
    
    results = []
    # Get unique circuit labels
    # circuit_labels = list(set(master_edge_list['direction'].unique()))

    for label in directions.keys():
        # Get unique neurons for this circuit
        circuit_edges = master_edge_list[master_edge_list['direction'] == label]
        path_ids = set(circuit_edges['pre_root_id']) | set(circuit_edges['post_root_id'])
        n_total_in_circuit = int(len(path_ids))
        
        # Filter neurons belonging to this circuit
        circuit_neurons = neurons[neurons['Root ID'].isin(path_ids)]
        
        # Create a dictionary to hold counts and stats for this specific circuit
        row = {
            'Circuit': directions[label],
            'Total Neurons': n_total_in_circuit
        }
        
        for nt in nt_types:
            n_nt_in_circuit = int(len(circuit_neurons[circuit_neurons['Predicted NT type'] == nt]))
            
            # Logic for the Plot: Concentration of this NT in this circuit
            concentration = n_nt_in_circuit / n_total_in_circuit if n_total_in_circuit > 0 else 0
            
            # We store the count or percentage to use for the bar height
            # Since you want a stacked bar where the total height of the bar 
            # represents the whole circuit, we use the count or the fraction.
            row[f'{nt} count'] = n_nt_in_circuit
            row[f'{nt} conc'] = concentration
        
        results.append(row)

    # Add the "All of BANC" summary row
    summary_row = {
        'Circuit': 'All of BANC',
        'Total Neurons': total_neurons
    }
    for nt in nt_types:
        n_nt_total = int(len(neurons[neurons['Predicted NT type'] == nt]))
        summary_row[f'{nt} count'] = n_nt_total
        summary_row[f'{nt} conc'] = n_nt_total / total_neurons if total_neurons > 0 else 0
    
    results.append(summary_row)

    df_enrichment = pd.DataFrame(results)
    return df_enrichment

# Run analysis
df_plot = enrichment_analysis()

# Plotting logic
# We use 'count' for the height of the segments. 
# If you want the bar to represent "percentage", you'd use 'conc'.
# Since each neuron has 1 NT, sum of counts = Total Neurons.

# Prepare data for Plotly
# The list of NT types to plot
nt_cols = [f'{nt} conc' for nt in neurons['Predicted NT type'].fillna("Unknown").unique()]
# The names for the legend
nt_names = [nt for nt in neurons['Predicted NT type'].fillna("Unknown").unique()]

fig = go.Figure()

for i, nt in enumerate(nt_names):
    fig.add_trace(go.Bar(
        name=nt_info[nt]["name"],
        x=df_plot['Circuit'],
        y=df_plot[nt_cols[i]],
        marker_color=nt_info[nt]["color"] # You can customize colors here
    ))

fig.update_layout(
    barmode='stack',
    title="Distribution of neurotransmitters across circuits",
    xaxis_title="Circuit",
    yaxis_title="Number of Neurons"
)

fig.show()